# AI Financial Risk Intelligence - RAG Pipeline\n
\n
This notebook implements the Retrieval-Augmented Generation (RAG) layer for the project described in the README.md. It follows the project architecture by ingesting financial documents, chunking them, embedding them, storing them in a vector database, and retrieving the most relevant context for answer generation.\n
\n
Key workflow:\n
- Load PDF and HTML documents\n
- Add organization metadata\n
- Chunk text for retrieval\n
- Create embeddings with Hugging Face\n
- Store embeddings in FAISS\n
- Retrieve relevant chunks\n
- Use an LLM to answer questions grounded in financial content\n

In [1]:
# !pip install --upgrade pip
# !pip install --force-reinstall "langchain>=0.2,<0.4" "langchain-community>=0.2,<0.4" "langchain-core>=0.2,<0.4" "langchain-huggingface" "langchain-openai" "langchain-text-splitters" "sentence-transformers" "transformers" "faiss-cpu" "pypdf" "beautifulsoup4"

In [1]:
from __future__ import annotations

import os
from pathlib import Path
from typing import List

from langchain_core.documents import Document
from langchain_community.document_loaders import BSHTMLLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_openai import ChatOpenAI
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

C:\Users\USER\AppData\Local\Temp\ipykernel_10732\3812703439.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import BSHTMLLoader, PyPDFLoader
c:\Users\USER\anaconda3\envs\ds\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
BASE_DIR = Path(r"C:\Users\USER\OneDrive\Desktop\AI-Financial-Risk-Intelligence")
DOCUMENT_DIR = BASE_DIR / "data" / "raw" / "Documents"
VECTOR_STORE_PATH = BASE_DIR / "vector_store" / "financial_rag_faiss"
VECTOR_STORE_PATH.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DOCUMENT_DIR:", DOCUMENT_DIR)
print("VECTOR_STORE_PATH:", VECTOR_STORE_PATH)

BASE_DIR: C:\Users\USER\OneDrive\Desktop\AI-Financial-Risk-Intelligence
DOCUMENT_DIR: C:\Users\USER\OneDrive\Desktop\AI-Financial-Risk-Intelligence\data\raw\Documents
VECTOR_STORE_PATH: C:\Users\USER\OneDrive\Desktop\AI-Financial-Risk-Intelligence\vector_store\financial_rag_faiss


In [ ]:
# ------------------------------------------------------------
# 2. Metadata helper
# ------------------------------------------------------------
def identify_source(filename: str) -> str:
    name = filename.lower()

    if "american" in name:
        return "American Express"
    if "rbi" in name:
        return "RBI"
    if "sebi" in name:
        return "SEBI"
    if "fatf" in name:
        return "FATF"
    if "basel" in name:
        return "Basel"
    if "jpmorgan" in name or "chase" in name:
        return "JPMorgan Chase"
    if "visa" in name:
        return "Visa"
    if "mastercard" in name:
        return "Mastercard"
    return "Unknown"


In [ ]:
# ------------------------------------------------------------
# 3. Load documents
# ------------------------------------------------------------
def load_pdf_documents(folder: str | Path) -> List[Document]:
    documents: List[Document] = []
    folder = Path(folder)

    for pdf_path in folder.rglob("*.pdf"):
        try:
            loader = PyPDFLoader(str(pdf_path))
            loaded_docs = loader.load()

            for doc in loaded_docs:
                doc.metadata["source_file"] = pdf_path.name
                doc.metadata["file_path"] = str(pdf_path)
                doc.metadata["document_type"] = "PDF"
                doc.metadata["organization"] = identify_source(pdf_path.name)

            documents.extend(loaded_docs)
        except Exception as exc:
            print(f"Failed to load PDF: {pdf_path} - {exc}")

    return documents

In [ ]:
def load_html_documents(folder: str | Path) -> List[Document]:
    documents: List[Document] = []
    folder = Path(folder)

    for html_path in folder.rglob("*.html"):
        try:
            loader = BSHTMLLoader(str(html_path))
            loaded_docs = loader.load()

            for doc in loaded_docs:
                doc.metadata["source_file"] = html_path.name
                doc.metadata["file_path"] = str(html_path)
                doc.metadata["document_type"] = "HTML"
                doc.metadata["organization"] = identify_source(html_path.name)

            documents.extend(loaded_docs)
        except Exception as exc:
            print(f"Failed to load HTML: {html_path} - {exc}")

    return documents


In [ ]:
pdf_documents = load_pdf_documents(DOCUMENT_DIR)
html_documents = load_html_documents(DOCUMENT_DIR)
documents = pdf_documents + html_documents

print("PDF documents loaded:", len(pdf_documents))
print("HTML documents loaded:", len(html_documents))
print("Total documents loaded:", len(documents))

if documents:
    for doc in documents[:2]:
        print("=" * 80)
        print("Source:", doc.metadata.get("source_file"))
        print("Organization:", doc.metadata.get("organization"))
        print(doc.page_content[:500])


Failed to load HTML: C:\Users\USER\OneDrive\Desktop\AI-Financial-Risk-Intelligence\data\raw\Documents\American_Express\DOC-02_American_Express_2026_Q1_10Q.html - By default BSHTMLLoader uses the 'lxml' package. Please either install it with `pip install -U lxml` or pass in init arg `bs_kwargs={'features': '...'}` to overwrite the default BeautifulSoup kwargs.
Failed to load HTML: C:\Users\USER\OneDrive\Desktop\AI-Financial-Risk-Intelligence\data\raw\Documents\American_Express\DOC-03_American_Express_2026_Q2_10Q.html - By default BSHTMLLoader uses the 'lxml' package. Please either install it with `pip install -U lxml` or pass in init arg `bs_kwargs={'features': '...'}` to overwrite the default BeautifulSoup kwargs.
Failed to load HTML: C:\Users\USER\OneDrive\Desktop\AI-Financial-Risk-Intelligence\data\raw\Documents\RBI\DOC-04_RBI_KYC_Master_Direction_2024.html - By default BSHTMLLoader uses the 'lxml' package. Please either install it with `pip install -U lxml` or pass in init arg `bs_k

In [ ]:
# ------------------------------------------------------------
# 4. Chunking
# ------------------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print("=" * 80)
    print("Chunk:", i)
    print("Source:", chunk.metadata.get("source_file"))
    print("Organization:", chunk.metadata.get("organization"))
    print(chunk.page_content[:350])

Total chunks created: 1457
Chunk: 0
Source: DOC-01_American_Express_2025_Annual_Report.pdf
Organization: American Express
2025 was an excellent year for American Express. Propelled 
by our loyal customers, our global network of merchants and 
partners, and our talented colleagues, we delivered some of 
the best financial results in our long history, building on the 
strong growth we have sustained since introducing our long-
term growth aspirations in 2022.
Our consis
Chunk: 1
Source: DOC-01_American_Express_2025_Annual_Report.pdf
Organization: American Express
can discover products and services, make decisions, 
and complete transactions on behalf of consumers and 
businesses – from booking travel and making dinner 
reservations to replenishing business inventories, managing 
expenses, and completing payments autonomously.
STEPHEN J. SQUERI, CHAIRMAN & CEO
Our business is driven by the value we create th
Chunk: 2
Source: DOC-01_American_Express_2025_Annual_Report.pdf
Organization: Ame

In [ ]:
# ------------------------------------------------------------
# 5. Embeddings + FAISS vector store
# ------------------------------------------------------------
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = FAISS.from_documents(chunks, embedding_model)
vectorstore.save_local(str(VECTOR_STORE_PATH))
print("Vector store saved at:", VECTOR_STORE_PATH)

loaded_vectorstore = FAISS.load_local(
    str(VECTOR_STORE_PATH),
    embedding_model,
    allow_dangerous_deserialization=True,
)

retriever = loaded_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)
print("Retriever ready. Top-k set to 5.")

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
'Server disconnected without sending a response.' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# ------------------------------------------------------------
# 6. LLM builder
# ------------------------------------------------------------
def build_llm():
    """Use OpenAI if configured, otherwise fallback to a local HF model."""
    if os.getenv("OPENAI_API_KEY"):
        return ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

    pipe = pipeline(
        "text2text-generation",
        model="google/flan-t5-small",
        max_new_tokens=256,
        device=-1,
    )
    return HuggingFacePipeline(pipeline=pipe)


llm = build_llm()



In [ ]:
prompt = ChatPromptTemplate.from_template(
    """
    You are a financial risk analyst assistant.
    Use only the context provided below to answer the user question.
    If the answer is not in the context, say so clearly and do not invent facts.
    Provide a concise but well-supported answer with cited source names when possible.

    Context:
    {context}

    Question:
    {input}
    """
)

In [ ]:
document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)
print("RAG chain built successfully.")

In [ ]:
# ------------------------------------------------------------
# 7. Run query
# ------------------------------------------------------------
query = "What are the key compliance and risk considerations in the financial documents?"
result = rag_chain.invoke({"input": query})

print("Question:", query)
print("\nAnswer:")
print(result["answer"])
print("\nRetrieved context sources:")
for idx, doc in enumerate(result["context"][:5], start=1):
    print(
        f"{idx}. {doc.metadata.get('source_file')} | "
        f"{doc.metadata.get('organization')} | "
        f"{doc.metadata.get('document_type')}"
    )

In [ ]:
# ------------------------------------------------------------
# 8. Reusable function
# ------------------------------------------------------------
def answer_question(question: str, k: int = 5):
    local_retriever = loaded_vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )

    local_llm = build_llm()

    local_prompt = ChatPromptTemplate.from_template(
        """
        You are a financial risk intelligence assistant.
        Answer the question using only the context.
        If the context does not contain the answer, say that clearly.

        Context:
        {context}

        Question:
        {input}
        """
    )

    local_document_chain = create_stuff_documents_chain(local_llm, local_prompt)
    local_rag_chain = create_retrieval_chain(local_retriever, local_document_chain)

    return local_rag_chain.invoke({"input": question})


# Example usage:
# resp = answer_question("Which regulator is mentioned in the compliance docs?", k=3)
# print(resp["answer"])

## Interactive PaySim Risk Investigation

Use the transaction form below with fields from the PaySim dataset. It starts with a real row from the local CSV, which you can edit before submitting. The saved PaySim model calculates the risk score and SHAP drivers; local FAISS retrieves evidence from `data/raw/Documents`; the report uses Hugging Face by default. Set `RISK_REPORT_PROVIDER=ollama` to use Ollama instead.

Run the next two code cells in order. The first loads the model and prepares the form; the second displays the form and runs the investigation.

In [1]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import clear_output, display

WORKSPACE_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "models" / "Fraud_models" / "paysim_fraud_model.pkl").is_file()
    ),
    None,
)
if WORKSPACE_ROOT is None:
    raise FileNotFoundError("Could not locate the project root containing the PaySim model.")
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from src.services.risk_investigation import (
    TRANSACTION_TYPES,
    generate_huggingface_report,
    generate_ollama_report,
    investigate_transaction,
    load_model_artifact,
)

MODEL_ARTIFACT = load_model_artifact(
    WORKSPACE_ROOT / "models" / "Fraud_models" / "paysim_fraud_model.pkl"
)
PAYSIM_CSV = (
    WORKSPACE_ROOT
    / "data"
    / "raw"
    / "Fraud"
    / "PaySim Financial Transactions"
    / "PS_20174392719_1491204439457_log.csv"
)
if not PAYSIM_CSV.is_file():
    raise FileNotFoundError(f"PaySim CSV not found: {PAYSIM_CSV}")

SAMPLE_TRANSACTION = pd.read_csv(PAYSIM_CSV, nrows=1).iloc[0]
STEP_INPUT = widgets.IntText(value=int(SAMPLE_TRANSACTION["step"]), description="Step")
TYPE_INPUT = widgets.Dropdown(
    options=TRANSACTION_TYPES,
    value=str(SAMPLE_TRANSACTION["type"]),
    description="Type",
)
AMOUNT_INPUT = widgets.FloatText(value=float(SAMPLE_TRANSACTION["amount"]), description="Amount")
ORIGIN_OLD_INPUT = widgets.FloatText(
    value=float(SAMPLE_TRANSACTION["oldbalanceOrg"]), description="Origin before"
)
ORIGIN_NEW_INPUT = widgets.FloatText(
    value=float(SAMPLE_TRANSACTION["newbalanceOrig"]), description="Origin after"
)
DEST_OLD_INPUT = widgets.FloatText(
    value=float(SAMPLE_TRANSACTION["oldbalanceDest"]), description="Destination before"
)
DEST_NEW_INPUT = widgets.FloatText(
    value=float(SAMPLE_TRANSACTION["newbalanceDest"]), description="Destination after"
)
RUN_ANALYSIS_BUTTON = widgets.Button(description="Analyze transaction", button_style="primary")
ANALYSIS_OUTPUT = widgets.Output()

print("Loaded model:", WORKSPACE_ROOT / "models" / "Fraud_models" / "paysim_fraud_model.pkl")
print("Loaded sample row from:", PAYSIM_CSV)
print("The form is initialized from the first local PaySim row; edit any values before analysis.")

c:\Users\USER\anaconda3\envs\ds\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Loaded model: c:\Users\USER\OneDrive\Desktop\AI-Financial-Risk-Intelligence\models\Fraud_models\paysim_fraud_model.pkl
Loaded sample row from: c:\Users\USER\OneDrive\Desktop\AI-Financial-Risk-Intelligence\data\raw\Fraud\PaySim Financial Transactions\PS_20174392719_1491204439457_log.csv
The form is initialized from the first local PaySim row; edit any values before analysis.


c:\Users\USER\OneDrive\Desktop\AI-Financial-Risk-Intelligence\src\services\risk_investigation.py:43: UserWarning: [20:54:20] WARNING: C:\Users\task_Mg8K6Y9nSmOox_0\croot\xgboost-split_1783526163892\work\src\gbm\../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  artifact = pickle.load(artifact_file)


In [2]:
from IPython.display import Markdown


def _run_payments_analysis(_):
    with ANALYSIS_OUTPUT:
        clear_output(wait=True)
        transaction = {
            "step": STEP_INPUT.value,
            "type": TYPE_INPUT.value,
            "amount": AMOUNT_INPUT.value,
            "oldbalanceOrg": ORIGIN_OLD_INPUT.value,
            "newbalanceOrig": ORIGIN_NEW_INPUT.value,
            "oldbalanceDest": DEST_OLD_INPUT.value,
            "newbalanceDest": DEST_NEW_INPUT.value,
        }
        provider = os.getenv("RISK_REPORT_PROVIDER", "huggingface").lower()
        if provider == "ollama":
            report_generator = generate_ollama_report
        elif provider == "huggingface":
            report_generator = generate_huggingface_report
        else:
            print("RISK_REPORT_PROVIDER must be 'huggingface' or 'ollama'.")
            return

        try:
            result = investigate_transaction(
                transaction,
                artifact=MODEL_ARTIFACT,
                generate_report=report_generator,
            )
        except Exception as exc:
            print(f"Investigation failed: {type(exc).__name__}: {exc}")
            return

        print("Model risk signal:", result["model"]["risk_level"])
        print("Fraud score:", f"{result['model']['fraud_risk_score']:.6f}")
        print("Model threshold:", f"{result['model']['threshold']:.6f}")
        print("Human review required:", result["human_review_required"])
        print("\nSHAP drivers:")
        for driver in result["shap_drivers"]:
            print(
                f"- {driver['feature']} = {driver['feature_value']:.4f}: "
                f"{driver['direction']} risk ({driver['shap_value']:+.4f})"
            )

        print("\nRetrieved regulatory evidence:")
        if not result["evidence"]:
            print("No evidence was retrieved. The report must state that evidence is insufficient.")
        for evidence in result["evidence"]:
            source = evidence["source"]
            if evidence["page"] is not None:
                source += f", page {evidence['page']}"
            print(f"\nSource: {source} | Organization: {evidence['organization']}")
            print(evidence["text"])

        print("\nInvestigation support report:")
        display(Markdown(result["report"]))


RUN_ANALYSIS_BUTTON.on_click(_run_payments_analysis)
display(
    widgets.VBox(
        [
            STEP_INPUT,
            TYPE_INPUT,
            AMOUNT_INPUT,
            ORIGIN_OLD_INPUT,
            ORIGIN_NEW_INPUT,
            DEST_OLD_INPUT,
            DEST_NEW_INPUT,
            RUN_ANALYSIS_BUTTON,
            ANALYSIS_OUTPUT,
        ]
    )
)